# نمونه‌گیری تصادفی از NYC High Volume FHV

این نوتبوک را در **Google Colab** باز کنید و از منوی **Runtime → Run all** اجرا کنید.

فایل‌های کامل روی ماشین ابری Colab دانلود می‌شوند، نه روی لپ‌تاپ شما. هر بار فقط یک فایل ماهانه نگه داشته می‌شود و داده به‌صورت دسته‌ای خوانده می‌شود تا کل ماه وارد RAM نشود. در پایان فقط فایل نمونه را دانلود کنید.

تنظیم پیش‌فرض: سال **۲۰۲۵**، هر ۱۲ ماه، **۱۰٬۰۰۰ ردیف تصادفی از هر ماه**؛ مجموعاً **۱۲۰٬۰۰۰ ردیف**. انتخاب در هر ماه تصادفی یکنواخت و بدون جایگذاری است. این یک نمونه‌گیری طبقه‌بندی‌شده بر اساس ماه است، نه نمونه‌گیری یکنواخت از کل سال. هیچ ردیفی پیش از نمونه‌گیری پاک یا فیلتر نمی‌شود.

منبع: [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)


In [ ]:
%pip -q install 'pyarrow>=14' numpy pandas


## تنظیمات

برای مشاهده اولیه یک ماه، `MONTHS = [1]` بگذارید. برای پوشش سال، تنظیم فعلی را نگه دارید. مقدار `SAMPLE_PER_MONTH` و `SEED` قابل تغییرند. ثابت نگه داشتن seed و فایل‌های منبع، نمونه را بازتولیدپذیر می‌کند.


In [ ]:
YEAR = 2025
MONTHS = list(range(1, 13))
SAMPLE_PER_MONTH = 10_000
SEED = 42
# None keeps all original columns. A list limits the output columns,
# but does NOT reduce the full-file download performed in this notebook.
COLUMNS = None


## دریافت روی Colab و نمونه‌گیری با حافظه محدود

هر بار یک ماه دریافت، نمونه‌گیری و سپس فایل اصلی حذف می‌شود. تعداد ردیف‌های واقعی هر ماه از metadata همان فایل خوانده می‌شود.


In [ ]:
import json
import csv
import random
import shutil
import tempfile
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


def uniform_sample_parquet(path, sample_n, seed, columns=None):
    if sample_n < 1:
        raise ValueError('sample_n must be positive')
    with pq.ParquetFile(path) as source:
        population_n = source.metadata.num_rows
        if population_n == 0:
            raise ValueError('The source file is empty')
        sample_n = min(sample_n, population_n)
        # Select actual row numbers uniformly across the ENTIRE month.
        chosen = np.array(sorted(random.Random(seed).sample(
            range(population_n), sample_n)), dtype=np.int64)
        pieces = []
        start = 0
        for batch in source.iter_batches(batch_size=65_536, columns=columns):
            stop = start + batch.num_rows
            left = int(np.searchsorted(chosen, start))
            right = int(np.searchsorted(chosen, stop))
            if left < right:
                local_indices = pa.array(chosen[left:right] - start)
                pieces.append(pa.Table.from_batches([batch.take(local_indices)]))
            start = stop
            if right == sample_n:
                break
    sample = pa.concat_tables(pieces)
    assert sample.num_rows == sample_n
    return sample, population_n, chosen


def save_samples(samples, records, output_dir, year, seed, columns):
    # Independent monthly files are authoritative: never cast or unify schemas.
    output_dir = Path(output_dir)
    monthly_dir = output_dir / 'monthly'
    monthly_dir.mkdir(parents=True, exist_ok=True)
    members, notices = [], []
    for sample, record in zip(samples, records):
        path = monthly_dir / f"sample_{record['month']}.parquet"
        pq.write_table(sample, path, compression='zstd', version='2.6',
                       coerce_timestamps=None, allow_truncated_timestamps=False,
                       store_schema=True)
        record['sample_file'] = str(path.relative_to(output_dir))
        record['schema'] = str(sample.schema)
        members.append(path)

    combined = None
    if all(s.schema.equals(samples[0].schema, check_metadata=True) for s in samples):
        combined = pa.concat_tables(samples, promote_options='none')
        permutation = np.random.default_rng(seed).permutation(combined.num_rows)
        combined = combined.take(pa.array(permutation))
        path = output_dir / 'sample.parquet'
        pq.write_table(combined, path, compression='zstd', version='2.6',
                       coerce_timestamps=None, allow_truncated_timestamps=False,
                       store_schema=True)
        members.append(path)
    else:
        notices.append('Monthly schemas differ. No merged Parquet was created; '
                       'all monthly samples retain their original column types.')

    # CSV is a viewing copy, not the lossless/type-preserving source of truth.
    # Write Arrow batches directly; never infer types via a pandas round trip.
    csv_path = output_dir / 'sample.csv'
    try:
        names = list(dict.fromkeys(name for s in samples for name in s.column_names))
        with csv_path.open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=names)
            writer.writeheader()
            for sample in ([combined] if combined is not None else samples):
                for batch in sample.to_batches(max_chunksize=1024):
                    writer.writerows(batch.to_pylist())
        members.append(csv_path)
    except Exception as exc:
        csv_path.unlink(missing_ok=True)
        notices.append(f'CSV preview could not be exported: {type(exc).__name__}: {exc}. '
                       'The monthly Parquet samples remain available.')

    manifest = {
        'method': 'Uniform sampling without replacement within each source monthly file',
        'year': year, 'seed': seed, 'columns': columns,
        'total_sample_rows': sum(s.num_rows for s in samples), 'months': records,
        'merged_parquet_created': combined is not None,
        'csv_created': csv_path.exists(), 'notices': notices,
        'notes': [
            'No cleaning, filtering, imputation, deduplication or type coercion.',
            'Monthly Parquet files are authoritative; read each file independently.',
            'CSV is only a viewing copy: types, null versus empty string, and absent '
            'versus null columns cannot be preserved by plain CSV.',
            'Equal monthly sample sizes require _sample_weight for annual trip-level estimates.',
            'Sampling metadata columns are additional; source columns are unchanged.',
            'Unreadable/corrupt source files raise an error; months are never silently skipped.',
            'Full source files were downloaded to the runtime, one at a time.'
        ]}
    manifest_path = output_dir / 'sampling_manifest.json'
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    members.append(manifest_path)
    zip_path = Path(f'hvfhv_{year}_random_sample.zip')
    # Publish a new archive only after every member has been written successfully.
    pending = output_dir / 'archive.pending'
    with zipfile.ZipFile(pending, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for member in members:
            archive.write(member, arcname=str(member.relative_to(output_dir)))
    pending.replace(zip_path)
    for notice in notices:
        print(notice)
    print(f"Complete: {manifest['total_sample_rows']:,} sampled rows; download {zip_path}")
    print(f'ZIP size: {zip_path.stat().st_size / 1_000_000:.2f} MB')
    return combined, pd.DataFrame(records), zip_path


def build_sample(year, months, sample_per_month, seed, columns=None):
    if not months or len(months) != len(set(months)):
        raise ValueError('Choose at least one month, without duplicates')
    if any(m < 1 or m > 12 for m in months):
        raise ValueError('Months must be between 1 and 12')
    samples, records = [], []
    with tempfile.TemporaryDirectory(prefix='hvfhv_source_') as temp_dir:
        for month in sorted(months):
            label = f'{year}-{month:02d}'
            url = ('https://d37ci6vzurychx.cloudfront.net/trip-data/'
                   f'fhvhv_tripdata_{label}.parquet')
            path = Path(temp_dir) / f'{label}.parquet'
            print(f'{label}: downloading to the cloud runtime...', flush=True)
            with urllib.request.urlopen(url, timeout=180) as response:
                if response.status != 200:
                    raise RuntimeError(f'Expected a complete file, got HTTP {response.status}')
                with path.open('wb') as destination:
                    shutil.copyfileobj(response, destination, length=1024 * 1024)
            original_bytes = path.stat().st_size
            try:
                sample, population_n, chosen = uniform_sample_parquet(
                    path, sample_per_month, seed + year * 100 + month, columns)
            except Exception as exc:
                raise RuntimeError(f'{label}: source could not be sampled. '
                                   'No automatic repair or skipped month.') from exc
            sample_n = sample.num_rows
            reserved = {'_sample_source_month', '_sample_source_row', '_sample_weight'}
            if reserved.intersection(sample.column_names) or len(set(sample.column_names)) != len(sample.column_names):
                raise ValueError(f'{label}: duplicate/reserved column name; refusing to overwrite source fields')
            sample = sample.append_column('_sample_source_month', pa.array([label] * sample_n))
            sample = sample.append_column('_sample_source_row', pa.array(chosen))
            sample = sample.append_column('_sample_weight', pa.array([population_n / sample_n] * sample_n))
            samples.append(sample)
            records.append({'month': label, 'population_rows': population_n,
                            'sample_rows': sample_n, 'source_bytes': original_bytes,
                            'sampling_weight': population_n / sample_n, 'source_url': url})
            path.unlink()
            print(f'{label}: {population_n:,} rows -> {sample_n:,} sampled rows', flush=True)
    # Unique run directory prevents stale outputs from entering a later archive.
    output_dir = Path(tempfile.mkdtemp(prefix=f'hvfhv_{year}_sample_', dir='.'))
    return save_samples(samples, records, output_dir, year, seed, columns)


In [ ]:
sample_table, monthly_counts, download_path = build_sample(
    YEAR, MONTHS, SAMPLE_PER_MONTH, SEED, COLUMNS)
display(monthly_counts[['month', 'population_rows', 'sample_rows', 'sampling_weight']])
print('Saved sample files:')
display(monthly_counts[['month', 'sample_file']])


## دانلود خروجی بدون پاک‌سازی

ZIP همیشه پوشهٔ `monthly/` شامل نمونه‌های Parquet هر ماه را دارد. این فایل‌ها مرجع اصلی‌اند: مقدار و نوع ستون‌های دادهٔ منبع حفظ می‌شود و فقط سه ستون اطلاعات نمونه‌گیری اضافه می‌شود. هر فایل ماهانه را جدا بخوانید؛ ابزارهای خواندن یک پوشه ممکن است دوباره نوع ستون‌ها را یکسان‌سازی کنند.

`sample.parquet` یکپارچه فقط اگر ساختار همهٔ ماه‌ها (شامل نوع ستون‌ها و metadata) یکسان باشد ساخته می‌شود؛ هیچ تبدیل نوع خودکاری انجام نمی‌شود. اختلاف ساختار در `sampling_manifest.json` ثبت می‌شود.

`sample.csv` فقط نسخهٔ قابل مشاهده است: CSV نوع ستون‌ها و تفاوت مقدار خالی با رشتهٔ خالی یا ستون ناموجود را حفظ نمی‌کند. اگر تبدیل به CSV خطا بدهد، خطا ثبت می‌شود و ZIP شامل نمونه‌های اصلی Parquet همچنان ساخته می‌شود.

مقادیر غلط، پرت، منفی، تکراری یا گمشده پاک‌سازی نمی‌شوند. فایل منبع خراب یا ناخوانا باعث توقف همراه خطای صریح می‌شود؛ هیچ ماهی بی‌صدا حذف نمی‌شود.

ستون‌های اضافه: `_sample_source_month` ماه فایل منبع، `_sample_source_row` شمارهٔ ردیف اصلی (از صفر)، `_sample_weight` وزن نمونه. برای آمار کل سال وزن‌ها را لحاظ کنید. این نمونه‌گیری برای بررسی اولیه است؛ کفایت آن برای زیرگروه‌های نادر باید جدا بررسی شود.


In [ ]:
from google.colab import files
files.download(str(download_path))
